# Can changing a suppressed activation subspace change the answer?

This notebook contains one example. Qwen3.5-4B first answers that the web-spinning animal
has 8 legs. We replace one residual component with a component extracted from a dog prompt.
At the displayed strength, the answer changes to 4.

This is a prompt-specific next-answer-state intervention, not evidence that dog identity was
transferred. `C=1` is the constructed replacement and still answers 8. The displayed `C=4`
result is a large extrapolation selected during earlier exploration.

In [1]:
import math
import os
from pathlib import Path
import subprocess
import sys

ROOT = Path(os.environ.get("SUPPRESSED_ROOT", ".")).resolve()
sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display
import torch
from tabulate import tabulate
from transformers import AutoModelForCausalLM, AutoTokenizer

from scripts.demo import generate, intervention_hooks, layer_hooks, run_forward, top_tokens, trajectory
from suppressed_activation_subspace import suppressed_activation_subspace

# Edit this cell. `just notebook-smoke` overrides the same values with environment variables.
MODEL = os.environ.get("SUPPRESSED_MODEL", "Qwen/Qwen3.5-4B")
REVISION = os.environ.get(
    "SUPPRESSED_REVISION", "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"
)
DEVICE = os.environ.get("SUPPRESSED_DEVICE", "cuda")
SOURCE_PROMPT = os.environ.get(
    "SUPPRESSED_SOURCE_PROMPT", "Fact: The number of legs on the animal that spins webs is "
)
TARGET_PROMPT = os.environ.get(
    "SUPPRESSED_DOG_PROMPT",
    "Fact: The number of legs on the animal that barks and is called man's best friend is ",
)
SOURCE_OUTPUT = os.environ.get("SUPPRESSED_SOURCE_OUTPUT", "8")
TARGET_OUTPUT = os.environ.get("SUPPRESSED_DOG_OUTPUT", "4")
EARLY_LAYER = int(os.environ.get("SUPPRESSED_EARLY_LAYER", 23))
PEAK_LAYER = int(os.environ.get("SUPPRESSED_PEAK_LAYER", 25))
OUTPUT_LAYER = int(os.environ.get("SUPPRESSED_OUTPUT_LAYER", 32))
INTERVENTION_LAYER = int(os.environ.get("SUPPRESSED_INTERVENTION_LAYER", 26))
RANK = int(os.environ.get("SUPPRESSED_RANK", 8))
STRENGTH = float(os.environ.get("SUPPRESSED_STRENGTH", 4))
GENERATION_TOKENS = int(os.environ.get("SUPPRESSED_TOKENS", 12))
GIT_DESCRIBE = subprocess.run(
    ["git", "describe", "--always", "--dirty"],
    cwd=ROOT,
    check=True,
    text=True,
    capture_output=True,
).stdout.strip()
{
    "git": GIT_DESCRIBE,
    "model": MODEL,
    "revision": REVISION,
    "extraction layers": (EARLY_LAYER, PEAK_LAYER, OUTPUT_LAYER),
    "intervention": f"one vector at residual L{INTERVENTION_LAYER}, final prompt token",
    "rank": RANK,
    "C": STRENGTH,
    "generation tokens": GENERATION_TOKENS,
}

/workspace/2026/suppressed-activations/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'git': 'v0.1.1-71-g3fb504c',
 'model': 'Qwen/Qwen3.5-4B',
 'revision': '851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a',
 'extraction layers': (23, 25, 32),
 'intervention': 'one vector at residual L26, final prompt token',
 'rank': 8,
 'C': 4.0,
 'generation tokens': 12}

Each prompt gets its own suppressed subspace from an unmodified forward pass. Extraction does not
receive *spider* or *dog* as a label:

```python
rise = logits[L25] - logits[L23]
fall = logits[L25] - logits[L32]
token_ids = topk(min(relu(rise), relu(fall)), rank=8)
S = qr(centered_lm_head[token_ids].T)
```

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
dtype = torch.bfloat16 if DEVICE == "cuda" else torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, dtype=dtype).to(DEVICE).eval()
language_model = model.model
blocks = language_model.layers
final_norm = language_model.norm
unembedding = model.lm_head.weight
norm_gain = 1.0 + final_norm.weight


def extract(prompt):
    input_ids = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(DEVICE)
    residuals, logits = trajectory(model, input_ids, final_norm)
    basis, selected = suppressed_activation_subspace(
        residuals[:, -1][None],
        unembedding,
        norm_gain,
        early_layer=EARLY_LAYER,
        peak_layer=PEAK_LAYER,
        output_layer=OUTPUT_LAYER,
        rank=RANK,
        normalize_unembedding_rows=True,
    )
    return {
        "input_ids": input_ids,
        "residuals": residuals,
        "logits": logits,
        "basis": basis[0],
        "selected": [tokenizer.decode([int(token_id)]) for token_id in selected[0]],
    }


def probability_table(logits, highlighted_token):
    rows = []
    for rank, row in enumerate(top_tokens(tokenizer, logits, k=10), start=1):
        token = row["token"]
        shown = f"**{token}**" if token == highlighted_token else token
        rows.append([rank, shown, f'{row["logp"]:.3f}', f'{math.exp(row["logp"]):.6f}'])
    return tabulate(
        rows,
        headers=["rank", "token", "log p", "p"],
        tablefmt="pipe",
        disable_numparse=True,
    )


source = extract(SOURCE_PROMPT)
target = extract(TARGET_PROMPT)
clean_generation = generate(
    model, tokenizer, source["input_ids"], blocks, {}, max_new_tokens=GENERATION_TOKENS
)
assert len(clean_generation["token_ids"]) == GENERATION_TOKENS

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 1577.99it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 426/426 [00:00<00:00, 9125.36it/s]

## Base

In [3]:
display(Markdown(f"""\
Input (`repr`, including the trailing space):

```python
{SOURCE_PROMPT!r}
```

Readout ("what it is thinking but not saying"):

```python
{source['selected']!r}
```

Generation (next {GENERATION_TOKENS} tokens, verbatim):

```text
{clean_generation['text']}
```

{probability_table(source['logits'], SOURCE_OUTPUT)}
"""))

Input (`repr`, including the trailing space):

```python
'Fact: The number of legs on the animal that spins webs is '
```

Readout ("what it is thinking but not saying"):

```python
['丝绸', '-web', 'Web', 'Disc', '的战', 'Spider', 'web', ' WEB']
```

Generation (next 12 tokens, verbatim):

```text
8.
Hypothesis: The animal that spins webs
```

| rank   | token   | log p   | p        |
|:-------|:--------|:--------|:---------|
| 1      | **8**   | -0.125  | 0.882568 |
| 2      | 4       | -2.875  | 0.056421 |
| 3      | 6       | -3.625  | 0.026651 |
| 4      | 1       | -4.625  | 0.009804 |
| 5      | 2       | -5.000  | 0.006738 |
| 6      | 3       | -5.125  | 0.005947 |
| 7      | 5       | -5.625  | 0.003607 |
| 8      | 7       | -5.750  | 0.003183 |
| 9      | 0       | -6.500  | 0.001504 |
| 10     | 9       | -6.562  | 0.001412 |


## Causal intervention

In [4]:
block = INTERVENTION_LAYER - 1  # decoder block output is the next residual
hook_by_layer = intervention_hooks(
    source["basis"],
    target["basis"],
    target["residuals"],
    operation="replace",
    strength=STRENGTH,
    blocks_to_hook=[block],
    prefill_only=True,
)
changed_logits = run_forward(model, source["input_ids"], blocks, hook_by_layer)
changed_generation = generate(
    model,
    tokenizer,
    source["input_ids"],
    blocks,
    hook_by_layer,
    max_new_tokens=GENERATION_TOKENS,
)
assert len(changed_generation["token_ids"]) == GENERATION_TOKENS

display(Markdown(f"""\
Input (`repr`, unchanged):

```python
{SOURCE_PROMPT!r}
```

Replacement readout ("what we insert"):

```python
{target['selected']!r}
```

Generation (next {GENERATION_TOKENS} tokens, verbatim):

```text
{changed_generation['text']}
```

{probability_table(changed_logits, TARGET_OUTPUT)}
"""))

Input (`repr`, unchanged):

```python
'Fact: The number of legs on the animal that spins webs is '
```

Replacement readout ("what we insert"):

```python
['吠', ' собаки', '狗粮', 'dog', 'Dog', ' Dog', 'สุนัข', ' canine']
```

Generation (next 12 tokens, verbatim):

```text
4.
Hypothesis: The animal that spins webs
```

| rank   | token   | log p   | p        |
|:-------|:--------|:--------|:---------|
| 1      | **4**   | -0.713  | 0.490091 |
| 2      | 8       | -1.213  | 0.297255 |
| 3      | 6       | -2.338  | 0.096505 |
| 4      | 1       | -3.213  | 0.040229 |
| 5      | 2       | -3.963  | 0.019003 |
| 6      | 5       | -4.088  | 0.016770 |
| 7      | 3       | -4.213  | 0.014799 |
| 8      | 9       | -4.338  | 0.013060 |
| 9      | 7       | -4.713  | 0.008976 |
| 10     | 0       | -6.588  | 0.001377 |


The replacement readout comes from an unmodified pass over this target input:

```python
"Fact: The number of legs on the animal that barks and is called man's best friend is "
```

For each complete input, an unmodified first pass extracts a separate subspace. We then change
one residual vector at L26 and the final source-input token:

```python
source = h_spider @ S_spider @ S_spider.T
target = h_dog @ S_dog @ S_dog.T
target = target * norm(source) / norm(target)
h_replaced = match_norm(h_spider + C * (target - source), h_spider)
```

At C=1, the constructed replacement still generates 8 first. The displayed C=4 intervention
extrapolates past that replacement. Re-running the detector after intervention still returns
spider-related rows, so this does not establish a semantic `spider → dog` swap. C=4 changes 72%
of the residual norm, 21 of 256 matched-random interventions have an equal or larger effect, and
a `2 + 2` target produces the same first-token change. See the [fixed run
report](../out/2026-09-05_211609_causal-confirmation/recovered_log.md).

<!-- Notebook written by PI/gpt-5.4 from Michael J. Clark's requested demo structure. -->